In [ ]:
!pip install dbrepo python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from dbrepo.RestClient import RestClient
from dotenv import load_dotenv
import os

load_dotenv()
client = RestClient(
    "https://test.dbrepo.tuwien.ac.at/",
    username=os.getenv("DBREPO_USER"),
    password=os.getenv("DBREPO_PASS")
)

DATABASE_ID = os.getenv("DB_ID") #'9fa181a9-de7c-4d44-b367-517a51f31351'  #os.getenv("DB_ID") #"cf27a11d-58e5-4693-856c-e8f3527e3394"

tables = client.get_tables(DATABASE_ID)
for t in tables:
    print(f"Table: {t.name}, ID: {t.id}")
    table_detail = client.get_table(DATABASE_ID, t.id)
    for col in table_detail.columns:
        print(f"  Column: {col.name}, ID: {col.id}")

Table: wastewater_data, ID: c91317e1-3568-451e-a496-ff76453b353f
  Column: city_name, ID: bd8bcd51-9e98-46a0-8200-1200d49b9ba5
  Column: ref_year, ID: 1934b160-eafb-4325-b597-718b7ca1aab8
  Column: metabolite_name, ID: 605faa0c-41c6-4e6e-9677-ace01697c793
  Column: daily_mean_concentration, ID: b948c8d4-1204-4897-822d-4d6296d921f8
Table: gdp_data, ID: fb47cfcc-9802-4b27-9665-e9b3b2faa756
  Column: nuts_code, ID: 981e7393-e469-4060-8d52-604ff8d96b87
  Column: ref_year, ID: c06a9d89-1c79-4946-949d-85de8da323a6
  Column: gdp, ID: d9532bea-f0ed-4641-a8c5-4c8c0074bff7
  Column: currency, ID: 8fb74397-a683-40c6-8859-329e5355adc4
Table: city_map, ID: 56f561d5-5d48-41cf-983d-b9404233aea5
  Column: nuts_code, ID: 989ce469-d5d1-49eb-a144-5e691a0626b7
  Column: city_name, ID: a3ec313a-5824-4480-86fa-3e759b8610e9


In [2]:
from dbrepo.api.dto import UpdateColumn

mappings = [
    {
        "table_name": "city_map",
        "column": "nuts_code",
        "concept_uri": "http://purl.org/linked-data/sdmx/2009/dimension#refArea"
    },
    {
        "table_name": "city_map",
        "column": "city_name",
        "concept_uri": "http://purl.obolibrary.org/obo/NCIT_C95378"
    },
    {
        "table_name": "gdp_data",
        "column": "ref_year",
        "concept_uri": "http://rs.tdwg.org/dwc/terms/year"        
    },
    {
        "table_name": "gdp_data",
        "column": "currency",
        "concept_uri": "http://purl.org/linked-data/sdmx/2009/attribute#currency",
        "unit_uri": "https://www.omg.org/spec/Commons/QuantitiesAndUnits/hasUnit"
    },
    {
        "table_name": "gdp_data",
        "column": "gdp",
        "concept_uri": "http://purl.org/linked-data/sdmx/2009/measure#obsValue",
        "unit_uri": "https://www.omg.org/spec/Commons/QuantitiesAndUnits/QuantityValue"
    },
    {
        "table_name": "wastewater_data",
        "column": "metabolite_name",
        "concept_uri": "http://purl.obolibrary.org/obo/CHEBI_23367"        
    },
    {
        "table_name": "wastewater_data",
        "column": "daily_mean_concentration",
        "concept_uri": "http://purl.allotrope.org/ontologies/process#AFP_0002800",
        "unit_uri" : "https://www.omg.org/spec/Commons/QuantitiesAndUnits/DerivedUnit"
    }
]

BASE_URL = "https://test.dbrepo.tuwien.ac.at"

for m in mappings:

    print("\n----------------------")
    print(f"{m['table_name']}.{m['column']}")

    table = next((t for t in tables if t.name == m["table_name"]), None)

    if not table:
        print("Table not found")
        continue

    table_detail = client.get_table(DATABASE_ID, table.id)

    col = next((c for c in table_detail.columns if c.name == m["column"]), None)

    if not col:
        print("Column not found")
        continue

    url = f'/api/v1/database/{DATABASE_ID}/table/{table.id}/column/{col.id}'

    response = client._wrapper(method="put", url=url, force_auth=True,
                                 payload=UpdateColumn(concept_uri=m["concept_uri"],
                                                      unit_uri=m.get("unit_uri", "None")))

    print("STATUS:", response.status_code)

    if response.ok:
        print("Success")
    else:
        print("Failed")
        print(response.text)


----------------------
city_map.nuts_code
STATUS: 202
Success

----------------------
city_map.city_name
STATUS: 202
Success

----------------------
gdp_data.ref_year
STATUS: 202
Success

----------------------
gdp_data.currency
STATUS: 202
Success

----------------------
gdp_data.gdp
STATUS: 202
Success

----------------------
wastewater_data.metabolite_name
STATUS: 202
Success

----------------------
wastewater_data.daily_mean_concentration
STATUS: 202
Success


## Brief explanation of used ontologies
The semantic mappings were implemented using statistical and biomedical ontologies, primarily SDMX, ChEBI, and NCIt. SDMX was selected for statistical and regional indicators because it is widely used by organizations such as Eurostat and OECD, while ChEBI and Allotrope ontologies were used for domain-specific chemical and analytical concepts related to wastewater epidemiology data.

## Checking Fields in Database

In [3]:
for t in tables:
    if t.name == "gdp_data":
        tab_id = t.id

In [4]:
tab = client.get_table(database_id = DATABASE_ID, table_id = tab_id)

In [5]:
dict(dict(tab)["columns"][3])

{'id': '8fb74397-a683-40c6-8859-329e5355adc4',
 'name': 'currency',
 'database_id': '5cde660e-153a-4bff-8e41-69e87cda399d',
 'table_id': 'fb47cfcc-9802-4b27-9665-e9b3b2faa756',
 'ord': 3,
 'internal_name': 'currency',
 'is_null_allowed': True,
 'type': <ColumnType.VARCHAR: 'varchar'>,
 'alias': None,
 'description': None,
 'size': 10,
 'd': None,
 'mean': None,
 'median': None,
 'concept': None,
 'unit': None,
 'concept_uri': 'http://purl.org/linked-data/sdmx/2009/attribute#currency',
 'unit_uri': 'https://www.omg.org/spec/Commons/QuantitiesAndUnits/hasUnit',
 'enums': [],
 'sets': [],
 'index_length': None,
 'length': None,
 'data_length': None,
 'max_data_length': None,
 'num_rows': None,
 'val_min': None,
 'val_max': None,
 'std_dev': None}